In [1]:
import pandas as pd
import os

/Users/tracyliu/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Data curation: Part I

##### 1. Combine dicomtocsv_series.csv and dicomtocsv_study.csv
##### 2. Group by PatientID

In [2]:
def merge_and_extract(file_path, file_series, file_study, output_file, extract_cols):
    """
    Merges two files into a single file based on shared columns, then extract desired columns
    """
    
    df_dicom_series = pd.read_csv(os.path.join(file_path, file_series))
    df_dicom_study = pd.read_csv(os.path.join(file_path, file_study))
    
    series_cols = df_dicom_series.columns
    study_cols = df_dicom_study.columns
    
    common_cols = series_cols.intersection(study_cols)
    series_not_study = series_cols.difference(study_cols)
    study_not_series = study_cols.difference(series_cols)
    
    df_all = pd.merge(df_dicom_series, df_dicom_study, how="outer", on=common_cols.to_list())
    df = df_all[extract_cols]
    
    if output_file:
        df.to_excel(output_file, index=False)
    
    return df_all, df

In [3]:
file_path = "../Data/Lee, Ju Hun's files - R3Data"

file_series = "dicomtocsv_series.csv"
file_study = "dicomtocsv_study.csv"
# output_file = "../Data/output.xlsx"

extract_cols = ['PatientID', 'AccessionNumber', 'PatientBirthDate', 'PatientAge', 'PatientSex',
        'StudyDate', 'StudyTime', 'AcquisitionDate',
        'Modality', 'Manufacturer', 'ManufacturerModelName',
        'StudyDescription', 'SeriesNumber', 'SeriesDescription',
        'Exposure',
        'Rows', 'Columns', 'PixelSpacing',
        'Modality', 'Manufacturer', 'ManufacturerModelName']

df_all, df = merge_and_extract(file_path, file_series, file_study, None, extract_cols)

In [4]:
def group_by_patient(df, output_file):
    """
    Group dataframe by patient ID
    """    
    PIDs = df["PatientID"].unique()
    num_patient = PIDs.size
    
    count = 0
    for pid in PIDs:
        if count == 0: 
            tmp = df[df["PatientID"] == pid]
        else: 
            tmp = pd.concat([tmp, df[df["PatientID"] == pid]], ignore_index=True)
        count += 1
    
#     tmp.sort_values(by=['PatientID', 'StudyDate'], inplace=True, ignore_index=True)
    tmp.sort_values(by=['PatientID', 'StudyDate', 'AccessionNumber'], inplace=True, ignore_index=True)
    
    if output_file:
        tmp.to_excel(output_file, index=False)
    
    return num_patient, tmp   

In [5]:
output_file = "../Data/dicom.xlsx"
num_patient, output = group_by_patient(df, None)

### Read

In [2]:
output_file = "../Data/dicom.xlsx"
dicom = pd.read_excel(output_file)

## Data curation: Part II

##### 1. For both control and cancer, combine files with the same name in different subfolders disregard differnt extension (e.g., .csv, .txt). Control and cancer stored separately

In [3]:
def combine_and_extract(shared_path, study, folder, file, extract_cols):
    """
    Combine files with the same name into a single file, then extract desired columns
    study: Control or Cancer
    folder: folders under Control or Cancer
    file: which files to process
    extract_cols (list): extracted columns based on file_param 
    """
    count = 0
    for f in folder:
        if count == 0:
            try:
                # for txt #
                df = pd.read_csv(os.path.join(shared_path, study, f, file) + ".txt", sep='|')
            except:
                # for csv #
                df = pd.read_csv(os.path.join(shared_path, study, f, file) + ".csv", encoding='latin-1')
            
        else:
            try:
                # for txt #
                tmp = pd.read_csv(os.path.join(shared_path, study, f, file) + ".txt", sep='|')
            except:
                # for csv #
                try:
                    tmp = pd.read_csv(os.path.join(shared_path, study, f, file) + ".csv", encoding='latin-1')
                except: 
                    print("N/A")
                
                
            try:
                df = pd.concat([df, tmp])
            except: 
                print("No file to concat: file not exist in " + os.path.join(shared_path, study, f))
        
        count += 1
        

#     # Remove duplicated entries
#     df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)
    # Sort by 'PATIENT_STUDY_ID
    df.sort_values(by='PATIENT_STUDY_ID', inplace=True, ignore_index=True)
        
    return df[extract_cols]

### Control/Cancer

<span style="color: red;">Choose **Control or Cancer**, change study, folder, and study_ext</span>

In [4]:
shared_path = "../Data/Lee, Ju Hun's files - R3Data"

In [29]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]
study_ext = "_controls"

# study = "Cancer"
# folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]
# study_ext = ""

### Files

See **parameter explanation for detail information

In [6]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [7]:
files

['pathology',
 'pathology_findings',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

#### 1. pathology

<span style="color: blue;">Change idx value **based on file**</span>

In [59]:
idx = 1
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

pathology
['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'SIDE']


In [60]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

N/A
No file to concat: file not exist in ../Data/Lee, Ju Hun's files - R3Data/Cancer/R3_3787_Lee_Data_Cancer_20240509


<span style="color: blue;">Reformat PATHOLOGY_DATE **based on file**</span>

In [61]:
df['PATHOLOGY_DATE'] = pd.to_datetime(df['PATHOLOGY_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [62]:
df[df.duplicated()]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
10,4330103113,425798,2018-01-29,Benign,R
23,4330214688,478250,2021-04-06,Benign,R
26,4330251952,462069,2021-12-15,Benign,R
32,4330268555,429934,2018-09-28,Benign,L
48,4330331505,422139,2018-05-09,Malignant,R
...,...,...,...,...,...
14030,4339566815,418774,2019-09-26,Benign,L
14048,4339601532,423569,2018-03-12,Benign,R
14059,4339778947,466925,2022-05-25,Benign,L
14066,4339933706,441680,2017-07-10,Benign,L


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [63]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [64]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', 'BX_ID'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [65]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-65-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 2. pathology_findings

<span style="color: blue;">Change idx value **based on file**</span>

In [89]:
idx = 2
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

pathology_findings
['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'PATHOLOGY_CD', 'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR', 'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'STAGE_NUM', 'MARGIN_STATUS']


In [90]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat PATHOLOGY_DATE **based on file**</span>

In [91]:
df['PATHOLOGY_DATE'] = pd.to_datetime(df['PATHOLOGY_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [92]:
df[df.duplicated()]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,MARGIN_STATUS
28,4333008024,410317,2019-06-17,BENIGN,AD,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,4333062366,475428,2020-08-06,BENIGN,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,4333062366,471969,2020-12-04,BENIGN,PA,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,4333101704,355101,2022-07-15,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,4333373719,471839,2020-10-29,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,4333373719,471830,2020-10-29,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,4333488305,417825,2019-11-07,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
173,4333514173,478396,2021-03-19,BENIGN,AN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177,4333571001,427540,2018-11-12,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191,4333624870,469223,2022-02-21,BENIGN,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [94]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [95]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', 'BX_ID'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [96]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-96-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 3. family_hx

<span style="color: blue;">Change idx value **based on file**</span>

In [97]:
idx = 3
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

family_hx
['PATIENT_STUDY_ID', 'LINE_NUM', 'CONTACT_DATE', 'MEDICAL_HX_TITLE', 'RELATION_TITLE']


In [98]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat CONTACT_DATE **based on file**</span>

In [99]:
df['CONTACT_DATE'] = pd.to_datetime(df['CONTACT_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [100]:
df[df.duplicated()]

,PATIENT_STUDY_ID,LINE_NUM,CONTACT_DATE,MEDICAL_HX_TITLE,RELATION_TITLE
526,4330000534,4,2018-10-25,ARTHRITIS,BIOLOGICAL MOTHER
527,4330000534,3,2018-10-25,RETINAL DETACHMENT,BIOLOGICAL MOTHER
528,4330000534,2,2018-10-25,HYPERTENSION,BIOLOGICAL MOTHER
529,4330000534,1,2018-10-25,MIGRAINES,SISTER
530,4330000534,9,2018-10-25,ARTHRITIS,OTHER
...,...,...,...,...,...
12570920,4339949064,4,2019-07-15,DIABETES,SISTER
12570922,4339949064,1,2019-07-15,DIABETES,BIOLOGICAL MOTHER
12570923,4339949064,3,2022-08-15,"CA, LIVER",BIOLOGICAL FATHER
12570924,4339949064,3,2019-12-03,"CA, LIVER",BIOLOGICAL FATHER


In [101]:
df[(df["PATIENT_STUDY_ID"] == 4330114580) & (df["LINE_NUM"] == 7)].sort_values(by='CONTACT_DATE')

,PATIENT_STUDY_ID,LINE_NUM,CONTACT_DATE,MEDICAL_HX_TITLE,RELATION_TITLE


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [102]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [103]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'CONTACT_DATE'], ascending=[True, False], inplace=True, ignore_index=True)

In [104]:
output_file = os.path.join("../Data/", study, file + ".txt")
df_Xdup.to_csv(output_file, header=True, index=None, sep=' ')

#### 4. patient_demo

<span style="color: blue;">Change idx value **based on file**</span>

In [106]:
idx = 4
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

patient_demo
['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE', 'RACE_TITLE', 'ETHNIC_TITLE']


In [107]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat BIRTH_DATE **based on file**</span>

In [108]:
df['BIRTH_DATE'] = pd.to_datetime(df['BIRTH_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [109]:
df[df.duplicated()]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE
33,4330115804,1973-07-01,FEMALE,WHITE,NaN
92,4330242158,1970-07-01,FEMALE,WHITE,NaN
153,4330312676,1973-07-01,FEMALE,WHITE,NOT SPECIFIED
171,4330318623,1960-07-01,FEMALE,WHITE,NaN
178,4330320873,1962-07-01,FEMALE,NOT SPECIFIED,NaN
...,...,...,...,...,...
49155,4337981130,1975-07-01,FEMALE,WHITE,NaN
49192,4339042307,1953-07-01,FEMALE,WHITE,NaN
49283,4339515366,1970-07-01,FEMALE,WHITE,NaN
49371,4339771691,1966-07-01,FEMALE,WHITE,NOT SPECIFIED


In [110]:
df[df["PATIENT_STUDY_ID"] == 4330170072]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [111]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [112]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'BIRTH_DATE'], ascending=[True, False], inplace=True, ignore_index=True)

In [113]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-113-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 5. vitals

<span style="color: blue;">Change idx value **based on file**</span>

In [114]:
idx = 5
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

vitals
['PATIENT_STUDY_ID', 'DATE_TAKEN', 'WEIGHT', 'WEIGHT_UNIT', 'HEIGHT', 'HEIGHT_UNIT', 'BMI']


In [115]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat date **based on file**</span>

In [116]:
df['DATE_TAKEN'] = pd.to_datetime(df['DATE_TAKEN'], format='%m/%d/%Y %H:%M:%S')
df['DATE_TAKEN'] = df['DATE_TAKEN'].dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [117]:
df[df.duplicated()]

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
25,4330000534,2018-10-06,68.0,KG,NaN,CM,25.00
29,4330000534,2018-10-05,NaN,KG,NaN,CM,25.00
113,4330082402,2017-10-26,2624.0,OZ,67.0,IN,25.69
169,4330109886,2017-10-10,2096.0,OZ,64.0,IN,22.49
172,4330109886,2022-07-04,2160.0,OZ,64.0,IN,23.17
...,...,...,...,...,...,...,...
1137415,4339949064,2022-11-18,1760.0,OZ,61.0,IN,20.78
1137416,4339949064,2020-06-24,1776.0,OZ,61.0,IN,20.97
1137418,4339949064,2019-07-15,1808.0,OZ,61.0,IN,21.35
1137421,4339949064,2022-08-15,1731.2,OZ,61.0,IN,20.44


In [118]:
df[df["PATIENT_STUDY_ID"] == 4330103113].sort_values(by='DATE_TAKEN')

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [119]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [120]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'DATE_TAKEN'], ascending=[True, False], inplace=True, ignore_index=True)

In [121]:
df_Xdup

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330000534,2022-04-19,2848.0,OZ,65.0,IN,29.62
1,4330000534,2021-06-29,2976.0,OZ,65.0,IN,30.95
2,4330000534,2021-03-02,2896.0,OZ,65.0,IN,30.12
3,4330000534,2020-08-27,2944.0,OZ,65.0,IN,30.62
4,4330000534,2020-03-02,2560.0,OZ,66.0,IN,25.82
...,...,...,...,...,...,...,...
859060,4339982047,2021-04-15,2864.0,OZ,64.0,IN,30.73
859061,4339982047,2018-10-17,2944.0,OZ,64.0,IN,31.58
859062,4339988199,2022-05-26,3072.0,OZ,61.0,IN,36.28
859063,4339988199,2022-04-21,3216.0,OZ,61.0,IN,37.98


<span style="color: blue;">Identify entry without BMI OR lack of either weight or height to calculate BMI</span>

In [122]:
df_Xbmi = df_Xdup[df_Xdup["BMI"].isna()]
df_Xbmi2 = df_Xbmi[df_Xbmi["WEIGHT"].isna() | df_Xbmi["HEIGHT"].isna()]
df_Xbmi_idx = df_Xbmi.index
df_Xbmi2_idx = df_Xbmi2.index

In [123]:
df_Xbmi_idx.equals(df_Xbmi2_idx)

True

In [124]:
df_Xbmi_idx

Index([    21,    548,    549,    878,   1173,   1376,   1377,   1378,   1379,
         2099,
       ...
       857804, 857854, 858036, 858479, 858482, 858579, 858746, 858929, 858930,
       859051],
      dtype='int64', length=3288)

<span style="color: blue;">Remove entries without BMI</span>

In [125]:
df_Xdup.drop(df_Xbmi_idx, inplace=True)
df_Xdup.drop(['WEIGHT', 'WEIGHT_UNIT', 'HEIGHT', 'HEIGHT_UNIT'], axis=1, inplace=True)

In [126]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-126-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 6. risk_factors

<span style="color: blue;">Change idx value **based on file**</span>

In [66]:
idx = 6
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

risk_factors
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'RISK_FACTOR_CD', 'RISK_FACTOR_NAME', 'RISK_SEQUENCE', 'EXAM_COMPLETED_DATE']


In [67]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Standardize date **based on file**</span>

In [68]:
df['EXAM_COMPLETED_DATE'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
df['EXAM_COMPLETED_DATE'] = df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [69]:
df[df.duplicated()]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE
73,4330103113,78547611,OM,"Family history of ovarian cancer in mother, si...",19,2018-01-29
74,4330103113,78547611,3,Very strong family history of breast cancer (m...,16,2018-01-29
78,4330103113,78547611,3,Very strong family history of breast cancer (m...,16,2018-01-29
79,4330103113,78547611,OM,"Family history of ovarian cancer in mother, si...",19,2018-01-29
80,4330103113,61304464,3,Very strong family history of breast cancer (m...,16,2020-01-18
...,...,...,...,...,...,...
512011,4339933706,71871349,Q,Post-menopausal patient,2,2017-07-10
512012,4339933706,454877701,Q,Post-menopausal patient,2,2021-11-17
512013,4339933706,70511846,1,"Weak family history of breast cancer (aunt, gr...",14,2017-07-25
512014,4339933706,70511801,1,"Weak family history of breast cancer (aunt, gr...",14,2017-07-28


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [70]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [71]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [72]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-72-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 7. enteredit_findings

<span style="color: blue;">Change idx value **based on file**</span>

In [30]:
idx = 7
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

enteredit_findings
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'COMPOSITION_NAME', 'FINDING_LOCATION', 'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE']


In [31]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Standardize date **based on file**</span>

In [32]:
df['EXAM_COMPLETED_DATE'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
df['EXAM_COMPLETED_DATE'] = df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

In [33]:
df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
0,4330000534,451457076,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-06-25
1,4330000534,63280481,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-06-20
2,4330000534,451457076,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-06-25
3,4330000534,78048053,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-03-28
4,4330000534,60687102,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-06-10
...,...,...,...,...,...,...,...
655090,4339988199,65403287,Scattered fibroglandular (25% - 50%),NaN,1 - Negative,N-Normal interval follow-up,2018-11-28
655091,4339988199,452549960,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-03-01
655092,4339988199,452549960,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-03-01
655093,4339988199,61697072,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-02-21


<span style="color: blue;">Drop row with NaN in below columns **based on file**</span>

In [34]:
df.dropna(subset=['COMPOSITION_NAME', 'FINDING_LOCATION', 'FINDING_CATEGORY', 'FINDING_REC'], how='all', inplace=True)

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [35]:
df[df.duplicated()]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
2,4330000534,451457076,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-06-25
5,4330000534,78048053,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-03-28
7,4330000534,63280481,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-06-20
10,4330000534,67674008,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-03
12,4330000534,64787403,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-05-18
...,...,...,...,...,...,...,...
655086,4339982047,450915296,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-10-12
655089,4339988199,67718778,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-02-25
655092,4339988199,452549960,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-03-01
655093,4339988199,61697072,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-02-21


##### Duplicated entries ####
Identify combinations that DO NOT have 2 entries and clean those up (remove complete duplicates for combinations that DO NOT have 2 entries)

In [36]:
# 1. Create a temporary key column (tuple of pid and asn) for efficient grouping
df['key'] = list(zip(df['PATIENT_STUDY_ID'], df['ACCESSION_NUMBER']))
# 2. Group by the key and count the occurrences
counts = df.groupby(['key']).size().reset_index(name='Count')
# 3. Identify the keys that match the criteria (Count = 2) and the keys that mismatch
match_keys = counts[counts['Count'] == 2]['key'].tolist()
mismatch_keys = counts[counts['Count'] != 2]['key'].tolist()
mismatch_keys_c1 = counts[counts['Count'] == 1]['key'].tolist()
# 4. Filter the original DataFrame to create the two subsets
match_data = df[df['key'].isin(match_keys)].drop(columns=['key'])
mismatch_data = df[df['key'].isin(mismatch_keys)].drop(columns=['key'])
mismatch_data_c1 = df[df['key'].isin(mismatch_keys_c1)].drop(columns=['key'])
# match_data = df[df['key'].isin(match_keys)]
# mismatch_data = df[df['key'].isin(mismatch_keys)]
# mismatch_data_c1 = df[df['key'].isin(mismatch_keys_c1)]
#5. Remove the duplicaates in mismatch_data by only keeping the first entry of duplicates
processed_mismatch_data = mismatch_data.drop_duplicates(subset=mismatch_data.columns.tolist(), keep='first').reset_index(drop=True)

In [37]:
df_Xdup = pd.concat([match_data, processed_mismatch_data], ignore_index=True)

<span style="color: cyan;">conflicting entries per patient visit (more than 2) </span>

In [38]:
# 1. Create a temporary key column (tuple of pid and asn) for efficient grouping
df_Xdup['key'] = list(zip(df_Xdup['PATIENT_STUDY_ID'], df_Xdup['ACCESSION_NUMBER']))
# 2. Group by the key and count the occurrences
counts = df_Xdup.groupby(['key']).size().reset_index(name='Count')
counts['Count'].unique()

array([2, 1, 3, 4])

In [39]:
# 3. Identify the keys that match the criteria (Count = 2) and the keys that mismatch
match_keys = counts[counts['Count'] == 2]['key'].tolist()
match_keys_c1 = counts[counts['Count'] == 1]['key'].tolist()
mismatch_keys_c4 = counts[counts['Count'] == 4]['key'].tolist()
mismatch_keys_c3 = counts[counts['Count'] == 3]['key'].tolist()
# 4. Filter the original DataFrame to create the two subsets
mismatch_data_c3 = df_Xdup[df_Xdup['key'].isin(mismatch_keys_c3)].drop(columns=['key'])
mismatch_data_c4 = df_Xdup[df_Xdup['key'].isin(mismatch_keys_c4)].drop(columns=['key'])

In [40]:
mismatch_data_c1 = df_Xdup[df_Xdup['key'].isin(mismatch_keys_c1)].drop(columns=['key'])

In [41]:
mismatch_data_c1

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
311430,4330000534,60687102,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-06-10
311431,4330002818,451184127,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-06-29
311432,4330002818,66384981,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-04-12
311433,4330022845,71056632,Scattered fibroglandular (25% - 50%),NaN,3 - Probably benign - short interval follow-up,F-U-Follow-up (Unilateral) at short interval (...,2017-05-31
311434,4330022845,71683845,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2017-12-20
...,...,...,...,...,...,...,...
409869,4339945405,62000814,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-10-16
409870,4339948318,68041649,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-10-16
409874,4339967297,78628246,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-04-26
409875,4339978786,72061302,Extremely dense (>75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-04-11


In [42]:
mismatch_data_c3

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
312662,4330783890,78739612,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-04-18
312665,4330783890,78739612,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,U-Ultrasound,2018-04-18
312666,4330783890,78739612,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,V-Spot magnification view(s),2018-04-18
314036,4333016341,73249376,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,U-Ultrasound,2016-11-25
314038,4333016341,73249376,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,S-Spot compression,2016-11-25
...,...,...,...,...,...,...,...
407765,4335958070,64633794,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,U-Ultrasound,2019-05-31
407767,4335958070,64633794,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-05-31
407969,4335984850,74717554,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2016-10-05
407971,4335984850,74717554,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2016-10-05


In [43]:
mismatch_data_c3["FINDING_REC"].unique()

array(['P-Additional projections', 'U-Ultrasound',
       'V-Spot magnification view(s)', 'S-Spot compression',
       'N-Normal interval follow-up', 'M-Magnification Views',
       'F-Follow-up at short interval (1-11 months)'], dtype=object)

In [44]:
mismatch_data_c4

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
317837,4333090716,74233681,Scattered fibroglandular (25% - 50%),NaN,0 - Need additional imaging evaluation,U-Ultrasound,2016-08-22
317840,4333090716,74233681,Scattered fibroglandular (25% - 50%),NaN,0 - Need additional imaging evaluation,S-Spot compression,2016-08-22
317841,4333090716,74233681,Scattered fibroglandular (25% - 50%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2016-08-22
317842,4333090716,74233681,Scattered fibroglandular (25% - 50%),NaN,1 - Negative,N-Normal interval follow-up,2016-08-22
335250,4333398902,74431525,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,M-Magnification Views,2016-07-26
335252,4333398902,74431525,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,U-Ultrasound,2016-07-26
335253,4333398902,74431525,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2016-07-26
335254,4333398902,74431525,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2016-07-26
362926,4333940217,73573355,Scattered fibroglandular (25% - 50%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2016-11-01
362928,4333940217,73573355,Scattered fibroglandular (25% - 50%),NaN,1 - Negative,N-Normal interval follow-up,2016-11-01


<span style="color: blue;">Change sort_values **based on file**</span>

In [45]:
df.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [46]:
df.drop('key', axis=1, inplace=True)

In [47]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df.to_excel(output_file, index=False)

<ipython-input-47-d516b8c82510>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df.to_excel(output_file, index=False)


#### 8. hormonal_mens

<span style="color: blue;">Change idx value **based on file**</span>

In [91]:
idx = 8
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

hormonal_mens
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'AGE_FIRST_USE', 'AGE_LAST_USE', 'DURATION', 'CURRENT_USE_IND', 'NEVER_USE_IND', 'AGE_MENARCHE', 'AGE_FIRST_LIVE_BIRTH', 'AGE_MENOPAUSE', 'AGE_HYSTERECTOMY', 'AGE_RIGHT_OVARY_REMOVAL', 'AGE_LEFT_OVARY_REMOVAL', 'PARITY_COUNT', 'PREGNANCY_COUNT', 'LAST_MENSTRUAL_DATE', 'MENSTRUAL_STATUS_CD', 'EXAM_COMPLETED_DATE']


In [92]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Standardize date **based on file**</span>

In [93]:
df['EXAM_COMPLETED_DATE'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
df['EXAM_COMPLETED_DATE'] = df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [94]:
df[df.duplicated()]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_FIRST_USE,AGE_LAST_USE,DURATION,CURRENT_USE_IND,NEVER_USE_IND,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE
1,4330018595,60695725,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
3,4330018595,60103700,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
5,4330018595,60103700,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
6,4330018595,60103700,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
7,4330018595,66064198,0,0,0,N,Y,16,30,49,0,0,0,2.0,3.0,NaN,POSTNAT,2021-06-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1273015,4339959661,62216536,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,11/05/2020 00:00:00,PRE,2019-10-28
1273016,4339959661,453550821,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,10/28/2021 00:00:00,PRE,2021-11-22
1273017,4339959661,453550821,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,10/28/2021 00:00:00,PRE,2021-11-22
1273018,4339959661,454612462,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,10/28/2021 00:00:00,PRE,2021-11-15


In [95]:
df[(df["PATIENT_STUDY_ID"] == 4330103113) & (df["ACCESSION_NUMBER"] == 79000052)].sort_values(by='EXAM_COMPLETED_DATE')

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_FIRST_USE,AGE_LAST_USE,DURATION,CURRENT_USE_IND,NEVER_USE_IND,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE
355,4330103113,79000052,25,26,0,N,N,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
356,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
357,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
360,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
361,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
365,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
393,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
403,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
404,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
405,4330103113,79000052,25,26,0,N,N,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [96]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [97]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [98]:
output_file = os.path.join("../Data/", study, file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-98-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


### Read

In [2]:
study = "Control"
file = "risk_factors"

In [3]:
output_file = os.path.join("../Data/", study, file + ".xlsx")

In [4]:
df = pd.read_excel(output_file)

In [5]:
df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE
0,4330000534,60228678,Z,Late child bearing (after 30),6,2020-06-01
1,4330000534,60228678,0,No family history of breast cancer,13,2020-06-01
2,4330000534,60687102,Z,Late child bearing (after 30),6,2020-06-10
3,4330000534,60687102,0,No family history of breast cancer,13,2020-06-10
4,4330000534,63280358,Z,Late child bearing (after 30),6,2019-06-20
...,...,...,...,...,...,...
714704,4339988199,67718778,Q,Post-menopausal patient,2,2021-02-25
714705,4339988199,67718778,Z,Late child bearing (after 30),6,2021-02-25
714706,4339988199,452549960,0,No family history of breast cancer,13,2022-03-01
714707,4339988199,452549960,Q,Post-menopausal patient,2,2022-03-01


### Listing out all possible parameters in respective file

In [12]:
file_path = "../Data/Lee, Ju Hun's files - R3Data/Cancer/R3_3787_Lee_Data_Cancer_20240509"

In [13]:
files = [f for f in os.listdir(file_path) if os.path.isfile(os.path.join(file_path, f))]

In [18]:
f = files[0][:-4]
df_csv = pd.read_csv(os.path.join(file_path, files[0]), encoding='latin-1')
df = pd.DataFrame({'Name': df_csv.columns.tolist()})
df.to_excel("../Data/parameters.xlsx", sheet_name=f, index=False)

N = len(files)
i = 2
with pd.ExcelWriter("../Data/parameters.xlsx") as writer:
    for file in files[1:]:
        f = file[:-4]
        df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')
        df = pd.DataFrame({'Name': df_csv.columns.tolist()})  
        df.to_excel(writer, sheet_name=f, index=False)
        print("Finished extracting columns from file: " + f + ", N=" + str(i) + ", remaining: " + str(N-i))
        i+=1

<ipython-input-18-c165a4e6fe96>:4: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df.to_excel("../Data/parameters.xlsx", sheet_name=f, index=False)
<ipython-input-18-c165a4e6fe96>:8: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  with pd.ExcelWriter("../Data/parameters.xlsx") as writer:


Finished extracting columns from file: patient_demo_registry, N=2, remaining: 25
Finished extracting columns from file: social_hx_tob, N=3, remaining: 24
Finished extracting columns from file: surgical_path_notes, N=4, remaining: 23
Finished extracting columns from file: enteredit_findings, N=5, remaining: 22
Finished extracting columns from file: social_hx_alc, N=6, remaining: 21


<ipython-input-18-c165a4e6fe96>:11: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')


Finished extracting columns from file: lab_results, N=7, remaining: 20
Finished extracting columns from file: problem_list, N=8, remaining: 19
Finished extracting columns from file: lab_sensitivity, N=9, remaining: 18
Finished extracting columns from file: pathology_findings, N=10, remaining: 17
Finished extracting columns from file: discharge_summary, N=11, remaining: 16
Finished extracting columns from file: procedures, N=12, remaining: 15
Finished extracting columns from file: encounter, N=13, remaining: 14
Finished extracting columns from file: recommendation_ie, N=14, remaining: 13


<ipython-input-18-c165a4e6fe96>:11: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')


Finished extracting columns from file: patient_data_ie, N=15, remaining: 12
Finished extracting columns from file: clinical_findings_ie, N=16, remaining: 11
Finished extracting columns from file: med_order, N=17, remaining: 10
Finished extracting columns from file: hormonal_mens, N=18, remaining: 9
Finished extracting columns from file: family_hx, N=19, remaining: 8
Finished extracting columns from file: risk_factors, N=20, remaining: 7


<ipython-input-18-c165a4e6fe96>:11: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')


Finished extracting columns from file: procedure_notes, N=21, remaining: 6
Finished extracting columns from file: vitals, N=22, remaining: 5
Finished extracting columns from file: order_result, N=23, remaining: 4
Finished extracting columns from file: med_fill, N=24, remaining: 3
Finished extracting columns from file: diagnosis, N=25, remaining: 2
Finished extracting columns from file: img_pathology, N=26, remaining: 1
Finished extracting columns from file: img_procedures, N=27, remaining: 0
